# 🕹️ <span style=color:dodgerblue>LLM for video game knowledge assistance<span>

This notebook is used for explanation purpose. Once the `docker compose up` command
is executed the entire application is ready to be used.

## 📔 <span style=color:gold>What is this notebook about?</span>
This notebook is intended to explain all the pipeline and provide an overview 
of the processes involved in the application (ingestion, monitoring, evaluation, etc).  
It's a complement to the [README.md](README.md) that shows the internal mechanisms
of the scripts.

In [1]:
import time

from llm import RAGClient
from opensearchpy import OpenSearch


from IPython.core.display import Markdown

## <span style=color:green>📂 Opensearch client creation</span>

We create an opensearch client. It is used for indexation and search 
(lexical, semantic, hybrid).  
The `RAGClient` simplifies the use of the RAG.

In [2]:
# Opensearch client connection to the running docker container.
opensearch_client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "Opensearch16admin#"),
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
)

# This client has a large set of functionalities that ease the use of the llm.
rag_client = RAGClient(
    opensearch_client,
    model="gemini-3.1-flash-lite",  # This is usually the best free model.
)

## <span style=color:lightsalmon>⛁ Ingestion</span>

This part showcases the ingestion process and how It works.  
To avoid the long waiting time this part is entirely optional and is not a pre-requisite to advance
to the other blocks of this notebook.  
It's just used for explanation purpose.

Before anything we call the helper function `setup_embedder` this will set the 
ML model for our vector/semantic search (and hybrid search).  
Note that the `setup_embedder` will return the `model_id` that was already set up when we called
`docker compose up`.

In [3]:
# Helper for downloading and preparing the ML embedder (convert text to vector).
from opensearch_utils import setup_embedder

In [4]:
# This will give us the model id for future use.
model_id = setup_embedder(opensearch_client)

Found existing model ID '2CdXtJ8BZhz468SkVWcJ' in state: DEPLOY_FAILED
Found existing model ID 'WtxntJ8Bz4lTtz-qEBKm' in state: DEPLOYED
✅ Model 'WtxntJ8Bz4lTtz-qEBKm' is active and deployed. Reusing it.


### 🎮 <span style=color:darkorchid>IGDB ingestion</span>

**You can skip this section if you want.**

> <span style=color:gold>⚠️ **Warning**</span>    
> If you want to execute the IGDB ingestion yourself, you must have and IGDB 
> developer key and an account.  
> Please follow the instructions in this link: https://api-docs.igdb.com/#getting-started.  
> Then write all your information in the `.env` file.

In [5]:
# We import this two helpers for ingestion
from ingest import IGDB, Wikipedia

In [6]:
igdb = IGDB(opensearch_client)
# igdb.download(index="igdb_small")

### <span style=color:deepskyblue>📄 Wikipedia ingestion</span>

**You can skip this section if you want.**

> For the wikipedia ingest, **you don't need to do any extra setup** (even if you 
> skipped igdb ingestion part).


We pull the wikipedia information from huggingface index. The data was updated 
on 


In [7]:
wikipedia = Wikipedia(opensearch_client)
# wikipedia.download(index="wikipedia_small")

## 🧪 <span style=color:forestgreen>Evaluation</span>

Here we will use our `Evaluator` class to execute each one of the steps:
1. Ground truth generation
2. Search evaluation & optimization.
    1. Perform the base evaluation itself.
    2. Optimize the boost values.
3. Tools use and final RAG's answer evaluation.

In [8]:
import pandas as pd
from evaluation import Evaluator

# Free llm model for the judge
evaluator = Evaluator(RAGClient(opensearch_client, model="gemma-4-31b-it"))

### 🎯 <span style=color:orangered>Ground truth generation</span>

In order to evaluate the search quality and model's performance we must have 
querys and a target variable to use as evaluation method.  
In this case, the target variable is the document id and the query will be a llm 
generated question based on the target document. 

For instance, if the target document is about Mario Kart, the llm
will generate `questions_per_doc` questions related to the document topic.

> <span style=color:yellow>⚠️ **Warning**</span>  
> The code blocks in this section create a very small dataset so as to
> not waste your tokens.  
> You may create a very large dataset if you want.  
> Keep in mind that there is a pre-generated large ground truth dataset, 
> so you don't need to create one from zero.

This should take $\sim 2\text{-}3 \ \text{min}$ 

In [17]:
rag_client.model = "gemma-4-31b-it"
igdb_ground_truth_small = evaluator.generate_ground_truth(
    index="igdb",
    questions_per_doc=3,
    num_docs=5,
    file_path="data/igdb_ground_truth_small.csv",
)

wikipedia_ground_truth_small = evaluator.generate_ground_truth(
    index="wikipedia",
    questions_per_doc=3,
    num_docs=5,
    file_path="data/wikipedia_ground_truth_small.csv",
)

rag_client.model = "gemini-3.1-flash-lite"


Generating GT:  40%|████      | 2/5 [00:05<00:06,  2.32s/it]

Schema violation: parsed output is None. Retrying...


Generating GT: 100%|██████████| 5/5 [00:14<00:00,  2.81s/it]


In [9]:
igdb_ground_truth_small = pd.read_csv("data/igdb_ground_truth_small.csv")
wikipedia_ground_truth_small = pd.read_csv("data/wikipedia_ground_truth_small.csv")

### 🔎 <span style=color:goldenrod>Search evaluation & optimization</span>

We must evaluate our search functions. Do they return the relevant documents?
For this we will use two metrics

If you use the full dataset this will take around $10$ min, you can bring a coffe ☕

In [10]:
num = 5  # Number of retrieved documents per search
igdb_score = {}
wikipedia_score = {}

evaluator.ground_truth = igdb_ground_truth_small
for search_type in ("lexical", "semantic", "hybrid"):
    time.sleep(0.2)
    hr_score, mrr_score, _ = evaluator.evaluate_search(
        index="igdb",
        search_type=search_type,
        num=num,
        max_workers=2,
    )

    igdb_score[search_type] = {"hr": hr_score, "mrr": mrr_score}

evaluator.ground_truth = wikipedia_ground_truth_small
for search_type in ("lexical", "semantic", "hybrid"):
    time.sleep(0.2)
    hr_score, mrr_score, _ = evaluator.evaluate_search(
        index="wikipedia",
        search_type=search_type,
        num=num,
        max_workers=2,
    )

    wikipedia_score[search_type] = {"hr": hr_score, "mrr": mrr_score}


Evaluating Search: 100%|██████████| 15/15 [00:01<00:00,  9.17it/s]


In [13]:
display(Markdown("# IGDB search score"))
display(pd.DataFrame(igdb_score))

display(Markdown("# Wikipedia search score"))
display(pd.DataFrame(wikipedia_score))

# IGDB search score

,lexical,semantic,hybrid
hr,0.733333,0.200000,0.733333
mrr,0.611111,0.166667,0.576667


# Wikipedia search score

,lexical,semantic,hybrid
hr,1.000000,0.666667,1.000000
mrr,0.955556,0.424444,0.946667


Now, let's test for the bigger datasets. This will take some time $\sim 10$ min 

In [14]:
# Loads the larger pre-made ground truths.
igdb_ground_truth = pd.read_csv("data/igdb_ground_truth.csv")
wikipedia_ground_truth = pd.read_csv("data/wikipedia_ground_truth.csv")

In [15]:
output = input("Proceed with full search evaluation (This will take some time) [y/n]: ")

if output == 'y':
    num = 5  # Number of retrieved documents per search
    igdb_score = {}
    wikipedia_score = {}


    evaluator.ground_truth = igdb_ground_truth
    for search_type in ("lexical", "semantic", "hybrid"):
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="igdb",
            search_type=search_type,
            num=num,
            max_workers=2,
        )

        igdb_score[search_type] = {"hr": hr_score, "mrr": mrr_score}

    evaluator.ground_truth = wikipedia_ground_truth
    for search_type in ("lexical", "semantic", "hybrid"):
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="wikipedia",
            search_type=search_type,
            num=num,
            max_workers=2,
        )

        wikipedia_score[search_type] = {"hr": hr_score, "mrr": mrr_score}


In [16]:
display("IGDB search score")
display(pd.DataFrame(igdb_score))

display("Wikipedia search score")
display(pd.DataFrame(wikipedia_score))

'IGDB search score'

,lexical,semantic,hybrid
hr,0.733333,0.200000,0.733333
mrr,0.611111,0.166667,0.576667


'Wikipedia search score'

,lexical,semantic,hybrid
hr,1.000000,0.666667,1.000000
mrr,0.955556,0.424444,0.946667


Now we want to use a `boost_dict` which is a way to increase or decrease the 
importance/relevance of different search fields. For instance we may set a boost of $2$
for the *name* field and $3$ for the *storyline*, so that the search will tend to prioritize
word match more in the *storyline* field and less in the *name* field.

Here our optimization method is pure brute force, we will only test in a given set of values (n dimensional grid) and take the combination that returns the best result. It's simply that.

We will perform the optimization on each index (wikipedia and igdb) separately.
Given that this takes a long time we will just test a very small set of possible values. 

In [17]:
from itertools import product

#### IGDB Index search optimization

Here we will test two search methods the lexical search and hybrid search.
We want to tune the boosting parameters so as to have the maximum performance.  
This should take $\sim 3$ min

In [18]:
# performance_df = pd.DataFrame()
results = []

# Search boosting optimization
# For IGDB index

evaluator.ground_truth = igdb_ground_truth_small
vals = (0, 1, 2)
iter = 0

# Pre-calculate total iterations for the universal tqdm bar
total_iterations = len(
    list(product(vals, vals, vals, ("lexical", "semantic", "hybrid")))
)

from tqdm.auto import tqdm

with tqdm(total=total_iterations, desc="Optimizing IGDB Search") as pbar:
    for i, j, k, search_type in product(
        vals, vals, vals, ("lexical", "semantic", "hybrid")
    ):
        # Skip redundant boost combinations (i == j == k) where they are not the baseline (1)
        if (search_type != "semantic" and i == j == k != 1) or (
            search_type == "semantic" and (i != 0 or j != 0 or k != 0)
        ):
            pbar.update(1)
            continue

        # To avoid the Memory Circuit Breaker, we introduce a sleep
        # and limit concurrency in evaluate_search
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="igdb",
            search_type=search_type,
            boost_dict={"name": i, "summary": j, "storyline": k},
            max_workers=2,  # Reduce concurrency to minimize memory pressure
        )

        results.append(
            {
                "name": i,
                "summary": j,
                "storyline": k,
                "hr": hr_score,
                "mrr": mrr_score,
                "type": search_type,
            }
        )

        
        time.sleep(0.2)
        iter = 0

        iter += 1
        pbar.update(1)

igdb_performance_df = pd.DataFrame(results)

Optimizing IGDB Search:   0%|          | 0/81 [00:00<?, ?it/s]

Evaluating Search: 100%|██████████| 15/15 [00:01<00:00,  8.96it/s]


In [18]:
df = igdb_performance_df.sort_values("mrr", ascending=False)
display(Markdown("## Lexical search result"))
display(df[df.type == "lexical"].head(5).round(3))

display(Markdown("## Semantic search result"))
display(df[df.type == "semantic"].head(5).round(3))

display(Markdown("## Hybrid search result"))
display(df[df.type == "hybrid"].head(5).round(3))

## Lexical search result

,name,summary,storyline,hr,mrr,type
5,0,1,0,0.8,0.683,lexical
23,1,1,0,0.8,0.683,lexical
47,2,2,0,0.8,0.680,lexical
49,2,2,1,0.8,0.680,lexical
25,1,1,1,0.8,0.639,lexical


## Semantic search result

,name,summary,storyline,hr,mrr,type
0,0,0,0,0.533,0.439,semantic


## Hybrid search result

,name,summary,storyline,hr,mrr,type
48,2,2,0,0.733,0.633,hybrid
50,2,2,1,0.733,0.633,hybrid
24,1,1,0,0.733,0.633,hybrid
6,0,1,0,0.733,0.633,hybrid
14,0,2,1,0.667,0.617,hybrid


#### Wikipedia index search optimization

In [19]:
evaluator.ground_truth = wikipedia_ground_truth_small

results = []
vals = (0, 1, 2, 3)
iter = 0

# Pre-calculate total iterations for the universal tqdm bar
total_iterations = len(list(product(vals, vals, ("lexical", "semantic", "hybrid"))))

from tqdm.auto import tqdm

with tqdm(total=total_iterations, desc="Optimizing Wikipedia Search") as pbar:
    for i, j, search_type in product(vals, vals, ("lexical", "semantic", "hybrid")):
        # Skip redundant boost combinations for non-lexical searches (semantic ignores boosts)
        # and skip symmetric combinations (i == j) where they are not the baseline (1)
        if (search_type != "semantic" and i == j != 1) or (
            search_type == "semantic" and (i != 0 or j != 0)
        ):
            pbar.update(1)
            continue

        # To avoid the Memory Circuit Breaker, we introduce a sleep
        # and limit concurrency in evaluate_search
        # time.sleep(0.2)
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="wikipedia",
            search_type=search_type,
            boost_dict={"title": i, "text": j},
            max_workers=2,  # Reduce concurrency to minimize memory pressure
        )

        results.append(
            {
                "title": i,
                "text": j,
                "hr": hr_score,
                "mrr": mrr_score,
                "type": search_type,
            }
        )

        time.sleep(0.2)

        iter += 1
        pbar.update(1)

wikipedia_performance_df = pd.DataFrame(results)

Optimizing Wikipedia Search:   0%|          | 0/48 [00:00<?, ?it/s]

Evaluating Search: 100%|██████████| 15/15 [00:01<00:00,  9.54it/s]



⚠️ Circuit Breaker hit in search! Retry 1/5. Waiting 5.926276797185274s...

⚠️ Circuit Breaker hit in search! Retry 1/5. Waiting 5.57165243987208s...


Evaluating Search: 100%|██████████| 15/15 [00:01<00:00,  9.41it/s]


In [20]:
df = wikipedia_performance_df.sort_values("mrr", ascending=False)
display(Markdown("## Lexical search"))
display(df[df.type == "lexical"].head(5).round(3))

display(Markdown("## Semantic search"))
display(df[df.type == "semantic"].head(5).round(3))

display(Markdown("## Hybrid search"))
display(df[df.type == "hybrid"].head(5).round(3))

## Lexical search

,title,text,hr,mrr,type
1,0,1,1.0,0.867,lexical
5,0,3,1.0,0.867,lexical
3,0,2,1.0,0.867,lexical
9,1,1,1.0,0.867,lexical
19,2,3,1.0,0.867,lexical


## Semantic search

,title,text,hr,mrr,type
0,0,0,0.267,0.172,semantic


## Hybrid search

,title,text,hr,mrr,type
6,0,3,0.867,0.817,hybrid
2,0,1,0.867,0.817,hybrid
26,3,2,0.867,0.817,hybrid
20,2,3,0.867,0.817,hybrid
14,1,3,0.867,0.817,hybrid


#### Final conclusion

Now that we tuned we can see which 

### 🔧 <span style=color:silver>Tools and final RAG answer evaluation</span>

Here is the final evaluation. We want to see the performance of our agent when
using the entire RAG system and the optimized boost values. So we use two 
sources of evaluation. The user's evaluation and a separate llm-judge evaluation. 
The judge will evaluate the tool usage and the final answer quality based on the 
ground truth. The user only evaluates the final answer with good or bad review.

> This will take some time ⏰. So we will only test with 5

In [20]:
judge = RAGClient(opensearch_client, model="gemini-3.1-flash-lite")
rag_client.model = "gemini-3.1-flash-lite"
evaluator.rag_client = rag_client
evaluator.ground_truth = igdb_ground_truth_small.iloc[:5,:]
evaluator.evaluate_agent(judge, index="igdb", max_workers=1)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Quota exceeded (429). Retrying in 20.96 seconds...
Successfully retrieved the content.


Evaluating: 100%|██████████| 1/1 [01:49<00:00, 109.27s/it]

Quota exceeded (429). Retrying in 80.46 seconds...Failed to process: Which retro fighting title fea... Error: Socket operation on non-socket
Quota exceeded (429). Retrying in 20.20 seconds...
Quota exceeded (429). Retrying in 40.81 seconds...
Successfully retrieved the content.
Batch completed. Saving to database.


In [15]:
from metrics import load_judge_feedback_data

judge_eval = load_judge_feedback_data()

def score_converter(x):
    match x:
        case "good":
            return 1
        case "average":
            return 0.5
        case "bad":
            return 0.0
        case _:
            return 0

score = judge_eval[["answer_score", "tool_score"]].map(score_converter)

print(score.sum()/len(score))


answer_score    0.500000
tool_score      0.642857
dtype: float64


In [16]:
evaluator.ground_truth = wikipedia_ground_truth_small.iloc[:5,:]
evaluator.evaluate_agent(judge, index="wikipedia", max_workers=1)

NameError: name 'wikipedia_ground_truth_small' is not defined

## 📊 <span style=color:gold>Monitoring</span>

The final step is the monitoring, we must see the **usage** (tokens and cost) 
and **performance** (reviews) of our agent.  
For this, we can open the *streamlit* app in this address http://localhost:8501,
Click to the 

In [17]:
response, _ = rag_client.rag(
    "Tell me about Dark Souls. Give me information from both indices."
)
display(Markdown(response))

NameError: name 'rag_client' is not defined

In [ ]:
print([usage.prompt_token_count for usage in rag_client.usage_history])
print([usage.candidates_token_count for usage in rag_client.usage_history])
print([usage.total_token_count for usage in rag_client.usage_history])

Here is not very pleasent to see, open the address http://localhost:8501